In [1]:
# 01_data_preparation.ipynb
# Builds the harmonised analytic dataset from KNHANES 2020-2024 (hn20_all..hn24_all).
# Defines the binary diabetes outcome, selects and documents predictors, applies
# adult (>=19y) and complete-case restrictions, and saves the analytic dataset,
# a predictor dictionary, and the sex-disaggregated prevalence table.
# All random operations use seed=42 for reproducibility.

import warnings, json
from pathlib import Path
import numpy as np
import pandas as pd
import pyreadstat

warnings.filterwarnings("ignore")
SEED = 42
np.random.seed(SEED)

# ---- Resolve project paths (works whether run from repo root or notebooks/) ----
ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
DATA = ROOT / "data"
TAB = ROOT / "results" / "tables"
FIG = ROOT / "results" / "figures"
for p in (TAB, FIG):
    p.mkdir(parents=True, exist_ok=True)

# ---- KNHANES survey files (one per wave) ----
FILES = {2020: "hn20_all", 2021: "hn21_all", 2022: "hn22_all",
         2023: "hn23_all", 2024: "hn24_all"}

# ---- Variable set: outcome components + predictors ----
# Outcome components (used only to build the label, not as predictors)
OUTCOME_SRC = ["HE_HbA1c", "HE_glu", "DE1_dg", "HE_DM_HbA1c", "DE1_32", "DE1_31"]
# Predictors (16): anthropometric, dietary, lifestyle, socioeconomic
PRED = ["age", "HE_BMI", "HE_wc", "HE_wt", "HE_ht",            # anthropometric + age
        "N_EN", "N_CHO", "N_SUGAR", "N_CHOL",                   # dietary intake
        "sm_presnt", "dr_month", "pa_aerobic",                  # lifestyle (binary)
        "L_BR_FQ", "edu", "ho_incm", "D_1_1"]                   # ordinal socioeconomic/behaviour
STRAT = ["sex"]
USE = OUTCOME_SRC + PRED + STRAT

# ---- Human-readable labels for the predictor dictionary ----
LABELS = {
    "age": "Age (years)", "HE_BMI": "Body mass index (kg/m^2)",
    "HE_wc": "Waist circumference (cm)", "HE_wt": "Body weight (kg)",
    "HE_ht": "Height (cm)", "N_EN": "Energy intake (kcal/day)",
    "N_CHO": "Carbohydrate intake (g/day)", "N_SUGAR": "Sugar intake (g/day)",
    "N_CHOL": "Cholesterol intake (mg/day)", "sm_presnt": "Current smoker (1=yes,0=no)",
    "dr_month": "Monthly alcohol use (1=yes,0=no)", "pa_aerobic": "Meets aerobic activity guideline (1=yes,0=no)",
    "L_BR_FQ": "Breakfast frequency (1=5-7/wk .. 4=0/wk)", "edu": "Education level (1=elementary..4=college+)",
    "ho_incm": "Household income quartile (1=lowest..4=highest)", "D_1_1": "Self-rated health (1=very good..5=very poor)",
}
# DiCE actionability tier for each predictor (used downstream in counterfactual stage)
TIER = {
    "age": "immutable", "HE_ht": "immutable", "edu": "immutable",
    "ho_incm": "immutable", "D_1_1": "immutable",
    "HE_BMI": "constrained", "HE_wc": "constrained", "HE_wt": "constrained",
    "N_EN": "actionable", "N_CHO": "actionable", "N_SUGAR": "actionable",
    "N_CHOL": "actionable", "sm_presnt": "actionable", "dr_month": "actionable",
    "pa_aerobic": "actionable", "L_BR_FQ": "actionable",
}

# ---- Load and stack all waves ----
frames = []
for yr, fn in FILES.items():
    df, _ = pyreadstat.read_sas7bdat(str(DATA / f"{fn}.sas7bdat"), usecols=USE)
    df["year"] = yr
    frames.append(df)
raw = pd.concat(frames, ignore_index=True)
print(f"Stacked {len(raw):,} records across waves {list(FILES)}")

# ---- Recode 'non-applicable / unknown' sentinel codes to missing (ordinal vars) ----
SENTINEL = {"L_BR_FQ": [8, 9], "D_1_1": [9], "edu": [9, 99]}
for c, codes in SENTINEL.items():
    raw.loc[raw[c].isin(codes), c] = np.nan

# ---- Define the binary diabetes outcome (explicit, ADA/KNHANES-consistent) ----
# diabetes = 1 if ANY of:
#   HbA1c >= 6.5%  |  fasting plasma glucose >= 126 mg/dL  |  physician diagnosis
#   |  on oral glucose-lowering medication  |  on insulin  |  KNHANES HbA1c-based status == diabetes
raw["diabetes"] = (
    (raw["HE_HbA1c"] >= 6.5) | (raw["HE_glu"] >= 126) |
    (raw["DE1_dg"] == 1) | (raw["DE1_32"] == 1) | (raw["DE1_31"] == 1) |
    (raw["HE_DM_HbA1c"] == 3)
).astype("float")

# ---- Apply inclusion criteria: adults >= 19y, then complete-case on model variables ----
adults = raw[raw["age"] >= 19].copy()
analytic = adults.dropna(subset=PRED + STRAT + ["diabetes"]).reset_index(drop=True)
analytic["sex"] = analytic["sex"].map({1: "Male", 2: "Female"})
analytic["diabetes"] = analytic["diabetes"].astype(int)

print(f"Adults (>=19y): {len(adults):,}")
print(f"Complete-case analytic sample: {len(analytic):,}  (predictors: {len(PRED)})")
print(f"Retention after listwise deletion: {len(analytic)/len(adults)*100:.1f}%")

# ---- Save analytic dataset (predictors + sex + outcome + year) ----
KEEP = PRED + ["sex", "diabetes", "year"]
analytic[KEEP].to_parquet(DATA / "analytic_dataset.parquet", index=False)
analytic[KEEP].to_csv(DATA / "analytic_dataset.csv", index=False)

# ---- Predictor dictionary ----
pdict = pd.DataFrame({
    "variable": PRED,
    "description": [LABELS[v] for v in PRED],
    "dice_tier": [TIER[v] for v in PRED],
})
pdict.to_csv(TAB / "predictor_dictionary.csv", index=False)
print("\nPredictor dictionary:")
print(pdict.to_string(index=False))

# ---- Sex-disaggregated prevalence table ----
rows = []
for sx in ["Male", "Female"]:
    s = analytic[analytic["sex"] == sx]
    rows.append([sx, len(s), (s.diabetes == 0).sum(), round((s.diabetes == 0).mean()*100, 1),
                 (s.diabetes == 1).sum(), round((s.diabetes == 1).mean()*100, 1)])
tot = analytic
rows.append(["Total", len(tot), (tot.diabetes == 0).sum(), round((tot.diabetes == 0).mean()*100, 1),
             (tot.diabetes == 1).sum(), round((tot.diabetes == 1).mean()*100, 1)])
prev = pd.DataFrame(rows, columns=["Sex", "Total_N", "NonDiabetic_n", "NonDiabetic_pct",
                                   "Diabetic_n", "Diabetic_pct"])
prev.to_csv(TAB / "table1_prevalence.csv", index=False)
print("\nTable 1 - Sex-disaggregated diabetes prevalence:")
print(prev.to_string(index=False))

# ---- Metadata for reproducibility ----
meta = {"seed": SEED, "waves": list(FILES), "n_adults": int(len(adults)),
        "n_analytic": int(len(analytic)), "n_predictors": len(PRED),
        "predictors": PRED, "outcome_definition": "HbA1c>=6.5 | FPG>=126 | Dx | meds | insulin | KNHANES-dm",
        "prevalence_pct": {"Male": float((analytic[analytic.sex=='Male'].diabetes.mean())*100),
                            "Female": float((analytic[analytic.sex=='Female'].diabetes.mean())*100),
                            "Total": float(analytic.diabetes.mean()*100)}}
with open(DATA / "dataset_metadata.json", "w") as f:
    json.dump(meta, f, indent=2)
print("\nSaved: analytic_dataset.parquet/csv, predictor_dictionary.csv, table1_prevalence.csv, dataset_metadata.json")


Stacked 34,640 records across waves [2020, 2021, 2022, 2023, 2024]
Adults (>=19y): 29,347
Complete-case analytic sample: 23,104  (predictors: 16)
Retention after listwise deletion: 78.7%



Predictor dictionary:
  variable                                     description   dice_tier
       age                                     Age (years)   immutable
    HE_BMI                        Body mass index (kg/m^2) constrained
     HE_wc                        Waist circumference (cm) constrained
     HE_wt                                Body weight (kg) constrained
     HE_ht                                     Height (cm)   immutable
      N_EN                        Energy intake (kcal/day)  actionable
     N_CHO                     Carbohydrate intake (g/day)  actionable
   N_SUGAR                            Sugar intake (g/day)  actionable
    N_CHOL                     Cholesterol intake (mg/day)  actionable
 sm_presnt                     Current smoker (1=yes,0=no)  actionable
  dr_month                Monthly alcohol use (1=yes,0=no)  actionable
pa_aerobic   Meets aerobic activity guideline (1=yes,0=no)  actionable
   L_BR_FQ        Breakfast frequency (1=5-7/wk .. 4=0